In [ ]:
import pandas as pd
import numpy as np
import requests
from tqdm.notebook import tqdm
import os

os.chdir("/home/cc/phd/KGEmbeddings")

# Read JSON file
e_map = pd.read_json("/home/cc/phd/KGEmbeddings/data/umls/entity_map.json", typ='series').to_dict()
r_map = pd.read_json("/home/cc/phd/KGEmbeddings/data/umls/rel_map.json", typ='series').to_dict()

# Invert mappings
inv_e_map = {v: k for k, v in e_map.items()}
inv_r_map = {v: k for k, v in r_map.items()}

umls = pd.read_csv("/home/cc/phd/KGEmbeddings/data/umls/train.csv", low_memory=False)
umls_r5 = umls[umls['relation_id'] == 5]

shared_tails = umls_r5.groupby('tail_id')['head_id'].nunique()
shared_tails = shared_tails[shared_tails > 1]  # tails with more than 1 head

shared_tails = shared_tails.sample(frac=1, random_state=77)  # shufflle series

In [ ]:
import pickle
import os
import torch
import random
import numpy as np
from tqdm.notebook import tqdm

from codes.query_solver import GeometricSolver
from codes.triplets import TripletsEngine

# PATH = "/home/marco_dossena/PHD/KGEmbeddings/"
PATH = "/home/cc/phd/KGEmbeddings/"
EMBEDDING_DIM = 512
DATA = "umls"
MODEL_NAME = "TransE"
# MODEL_PATH = "/home/cc/phd/KGEmbeddings/models/TransE_FB15k_0/"
# MODEL_PATH = "/home/cc/phd/KGEmbeddings/models/RotatE_FB15k_0/"
MODEL_PATH = f"{PATH}models/{MODEL_NAME}_{DATA}_0"
# DICTS_DIR = "/home/cc/phd/KGEmbeddings/data/FB15k/"
DICTS_DIR = f"{PATH}data/{DATA}"

with open(f'queries/{DATA}/queries-big.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)

# queries = loaded_dict['queries']
# results = loaded_dict['results']

kg = TripletsEngine(os.path.join(DICTS_DIR), ext="txt" if DATA == "FB15k" else "csv", from_splits=True)
qs = GeometricSolver(MODEL_PATH, MODEL_NAME.lower(), EMBEDDING_DIM, h2t=kg.h2t, t2h=kg.t2h, k_neighbors=50, k_results=25, device='cuda')

In [ ]:
def api_call(cui):
    url = f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}?apiKey=f72ff16d-f1da-40a6-adbc-9f42ff7c9fe7"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data.get('result', {}).get('name', 'N/A')
    else:
        return 'N/A'

def print_query(query, res):
    h1, r1 = query[0][0]
    h2, r2 = query[0][1]
    last_rel = query[1]
    print(f"({api_call(inv_e_map[h1])} - [{inv_r_map[r1]}]-> ?)")
    print(f"AND ({api_call(inv_e_map[h2])} - [{inv_r_map[r2]}]-> ?)")
    print(f"AND (?* - [{inv_r_map[last_rel]}]-> ?')")
    print(f"Results: {[api_call(inv_e_map[r]) for r in res]}")

In [ ]:
queries = []   # to store query structure
results = []   # to store results per rel_target

if shared_tails.empty:
    print("No shared tails found with relation_id = 5")
else:
    for shared_tail_id in tqdm(shared_tails.index[:3000], desc="Processing queries"):
        # Get the heads pointing to this shared tail
        heads = umls_r5[umls_r5['tail_id'] == shared_tail_id]['head_id'].unique()[:2]
        if len(heads) < 2:
            continue  # need at least 2 heads

        # Save query structure
        query = [[(heads[0], 5), (heads[1], 5)]]

        # --- Step 2: new head = shared_tail_id ---
        new_head_id = shared_tail_id
        new_edges = umls[(umls['head_id'] == new_head_id) & (umls['relation_id'] != 0)]

        if new_edges.empty:
            continue
        
        # Group tails by relations
        relation_dict = (
            new_edges.groupby('relation_id')['tail_id']
            .apply(list)
            .to_dict()
        )
        
        for rel, tails in relation_dict.items():
            queries.append(query+[rel])
            results.append(tails)

In [ ]:
# TODO: Fix query execution: the results is one set for query!

def recall_at_k(pred, true, k):
    if len(true) == 0:
        return 1.0
    
    if k > 0:
        pred_k = pred[:max(k, len(true))]
    else:
        pred_k = pred

    hits = sum([1 for p in pred_k if p in true])
    return hits / len(true)

def map_at_k(pred, true, k):
    if len(true) == 0:
        return 1.0
    
    if k > 0:
        pred_k = pred[:max(k, len(true))]
    else:
        pred_k = pred

    hits = sum([1 for p in pred_k if p in true])
    return hits / max(k, len(true))

qs.set_k(k_neighbors=50, k_results=25)
recalls = {
    "recall": [],
    "recall5": [],
    "recall10": [],
    "recall25": [],
    "recall50": [],
}

maps = {
    'MAP@5': [],
    'MAP@10': [],
    'MAP@25': [],
    'MAP@50': [],
}

for query, result in tqdm(zip(queries, results), total=len(queries)):

    # result = list(result)

    res = qs.execute_query(query, proj_mode="inter", agg_mode="union", trues=result)

    if len(res) > 0:
        for k in [5, 10, 25, 50]:
            recalls[f"recall{k}"].append(recall_at_k(res, result, k))
            maps[f'MAP@{k}'].append(map_at_k(res, result, k))

        recalls["recall"].append(recall_at_k(res, result, 0))

metrics = qs.get_metrics()

print(f"Average Recall over {len(queries)} complex queries (2p1): {np.mean(recalls['recall'])}")
print(f"Average MRR over {len(queries)} complex queries (2pi): {np.mean(metrics['mrr'])}")
print(f"Average Recall@K over {len(queries)} complex queries (2p1): 5: {np.mean(recalls['recall5'])}, 10: {np.mean(recalls['recall10'])}, \
25: {np.mean(recalls['recall25'])}, 50: {np.mean(recalls['recall50'])}")
print(f"Average Hits@K over {len(queries)} complex queries (2pi): 1: {np.mean(metrics['hits1'])}, 3: {np.mean(metrics['hits3'])}, \
5: {np.mean(metrics['hits5'])}, 10: {np.mean(metrics['hits10'])}, 25: {np.mean(metrics['hits25'])}")
print(f"Average MAP@K over {len(queries)} complex queries (2p1): 5: {np.mean(maps['MAP@5'])}, 10: {np.mean(maps['MAP@10'])}, \
25: {np.mean(maps['MAP@25'])}, 50: {np.mean(maps['MAP@50'])}")

In [ ]:
idx = np.random.randint(0, len(queries)-1)

res = qs.execute_query(queries[idx], proj_mode="inter", agg_mode="union", trues=results[idx])

print_query(queries[idx], results[idx])

print(results[idx])
print(res)